In [2]:
import os
import string
import csv

# Paths
DATA_DIR = "../data"
CAPTIONS_FILE = os.path.join(DATA_DIR, "captions.txt")

def clean_caption(text):
    """
    Cleans a raw caption:
    - Converts to lowercase
    - Removes punctuation
    - Removes words containing digits
    - Removes single-letter hanging characters (keeps 'a')
    """
    text = str(text).lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    words = text.split()
    words = [w for w in words if w.isalpha() and (len(w) > 1 or w == "a")]
    return " ".join(words)

# Read and clean captions
records = []
with open(CAPTIONS_FILE, "r", encoding="utf-8") as f:
    reader = csv.reader(f)
    header = next(reader)  # Skip header: image,caption
    for row in reader:
        if len(row) >= 2:
            img_name = row[0].strip()
            # In case captions contain commas, rejoin remainder
            caption_raw = ",".join(row[1:]).strip()
            cleaned = clean_caption(caption_raw)
            seq = f"startseq {cleaned} endseq"
            records.append({
                "image": img_name,
                "caption": caption_raw,
                "caption_seq": seq
            })

print(f"Total captions processed: {len(records)}")

# Display the first 10 rows (equivalent to df.head(10))
print(f"\n{'Image':<27} | {'Cleaned Caption with Tokens'}")
print("-" * 80)
for item in records[:10]:
    print(f"{item['image']:<27} | {item['caption_seq']}")

Total captions processed: 40455

Image                       | Cleaned Caption with Tokens
--------------------------------------------------------------------------------
1000268201_693b08cb0e.jpg   | startseq a child in a pink dress is climbing up a set of stairs in an entry way endseq
1000268201_693b08cb0e.jpg   | startseq a girl going into a wooden building endseq
1000268201_693b08cb0e.jpg   | startseq a little girl climbing into a wooden playhouse endseq
1000268201_693b08cb0e.jpg   | startseq a little girl climbing the stairs to her playhouse endseq
1000268201_693b08cb0e.jpg   | startseq a little girl in a pink dress going into a wooden cabin endseq
1001773457_577c3a7d70.jpg   | startseq a black dog and a spotted dog are fighting endseq
1001773457_577c3a7d70.jpg   | startseq a black dog and a tricolored dog playing with each other on the road endseq
1001773457_577c3a7d70.jpg   | startseq a black dog and a white dog with brown spots are staring at each other in the street endseq
10

In [3]:
descriptions = {}
for item in records:
    img = item["image"]
    seq = item["caption_seq"]
    if img not in descriptions:
        descriptions[img] = []
    descriptions[img].append(seq)

print(f"Total unique images mapped: {len(descriptions)}")

Total unique images mapped: 8091


In [4]:
from collections import Counter

# Count all word occurrences across all captions
word_counts = Counter()
for caps in descriptions.values():
    for c in caps:
        # Exclude startseq and endseq when checking natural vocabulary counts
        words = c.split()[1:-1]
        word_counts.update(words)

total_words = sum(word_counts.values())
unique_words = len(word_counts)

print(f"Total word tokens: {total_words}")
print(f"Unique words in raw vocabulary: {unique_words}")

# Evaluate rare word thresholds
THRESHOLD = 5
vocab_filtered = [w for w, cnt in word_counts.items() if cnt >= THRESHOLD]
print(f"Unique words with count >= {THRESHOLD}: {len(vocab_filtered)}")
print(f"Reduction: dropped {unique_words - len(vocab_filtered)} rare words.")

Total word tokens: 435887
Unique words in raw vocabulary: 8764
Unique words with count >= 5: 2982
Reduction: dropped 5782 rare words.


In [6]:
# Compute sequence lengths including startseq and endseq
caption_lengths = [len(c.split()) for caps in descriptions.values() for c in caps]
max_seq_len = max(caption_lengths)
avg_seq_len = sum(caption_lengths) / len(caption_lengths)

print(f"Maximum sequence length: {max_seq_len} words")
print(f"Average sequence length: {avg_seq_len:.2f} words")


Maximum sequence length: 37 words
Average sequence length: 12.77 words


In [7]:
import json

# 1. Save cleaned captions dictionary
cleaned_captions_file = os.path.join(DATA_DIR, "cleaned_captions.json")
with open(cleaned_captions_file, "w", encoding="utf-8") as f:
    json.dump(descriptions, f, indent=2)

# 2. Save filtered vocabulary words
vocab_file = os.path.join(DATA_DIR, "vocab.txt")
with open(vocab_file, "w", encoding="utf-8") as f:
    for word in sorted(vocab_filtered):
        f.write(f"{word}\n")

print(f"Saved: {cleaned_captions_file}")
print(f"Saved: {vocab_file}")

Saved: ../data/cleaned_captions.json
Saved: ../data/vocab.txt


In [8]:
# 0 is traditionally reserved for padding in sequence models
PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"

# Build the word-to-index mapping
word_to_idx = {PAD_TOKEN: 0, UNK_TOKEN: 1}

# Start regular vocabulary indexing from 2
for idx, word in enumerate(sorted(vocab_filtered), start=2):
    word_to_idx[word] = idx

# Also build the reverse mapping (index-to-word) for caption generation later
idx_to_word = {idx: word for word, idx in word_to_idx.items()}

vocab_size = len(word_to_idx)
print(f"Total vocabulary size (including special tokens): {vocab_size}")
print(f"Index for 'startseq': {word_to_idx.get('startseq')}")
print(f"Index for 'endseq': {word_to_idx.get('endseq')}")
print(f"Index for 'dog': {word_to_idx.get('dog')}")


Total vocabulary size (including special tokens): 2984
Index for 'startseq': None
Index for 'endseq': None
Index for 'dog': 740


In [10]:
import json

# 1. Save cleaned captions dictionary (image -> list of 5 captions)
captions_save_path = os.path.join(DATA_DIR, "cleaned_captions.json")
with open(captions_save_path, "w", encoding="utf-8") as f:
    json.dump(descriptions, f, indent=2)

# 2. Save word_to_idx mapping
word_to_idx_path = os.path.join(DATA_DIR, "word_to_idx.json")
with open(word_to_idx_path, "w", encoding="utf-8") as f:
    json.dump(word_to_idx, f, indent=2)

# 3. Save metadata (vocab size, max sequence length)
metadata = {
    "vocab_size": vocab_size,
    "max_seq_len": max_seq_len,
    "threshold": THRESHOLD
}
metadata_path = os.path.join(DATA_DIR, "dataset_metadata.json")
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Saved files successfully:")
print(f" - {captions_save_path}")
print(f" - {word_to_idx_path}")
print(f" - {metadata_path}")

Saved files successfully:
 - ../data/cleaned_captions.json
 - ../data/word_to_idx.json
 - ../data/dataset_metadata.json
